# Notebook 10 — Interpretability: What Is the Model Actually Looking At?

You trained a model and the validation accuracy is great. But *why* does it predict what it predicts? Did it learn the object, or did it learn a spurious shortcut (grass = 'cow', snow = 'wolf', watermark = 'airplane')? In this notebook we use **Grad-CAM** to visualize exactly which pixels drive each decision, then walk through a small debugging workflow that catches bad models before they ship.

## Learning goals
- Understand three concrete reasons interpretability matters (debugging, trust, compliance).
- Derive Grad-CAM from scratch in math and then call `utils.gradcam.GradCAM`.
- Visualize which regions activate for a predicted class on real Pet images.
- Meet Grad-CAM++ and Score-CAM via the `pytorch-grad-cam` library.
- Use Grad-CAM to diagnose misclassifications.
- Visualize raw feature maps as a complementary lens.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/10_interpretability_gradcam.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device

## 1. Why interpretability?

Three reasons — any one is enough to justify the effort:

1. **Debugging.** When a model disagrees with your expectations, an explanation tells you whether the dataset is wrong, the architecture is wrong, or the training is incomplete. It's a huge accelerator for fixing models.
2. **Trust.** Stakeholders — radiologists, regulators, your PM — need to believe the model before deploying it. An attribution map turns a black box into a first-class argument.
3. **Compliance.** The EU AI Act, US FDA guidance on AI/ML SaMD, and internal AI governance committees increasingly *require* post-hoc explanations for high-risk decisions. Interpretability is no longer optional for regulated domains.

Grad-CAM is the most common technique for CNN image classifiers. It is not the only one — SHAP, integrated gradients, LIME, attention rollout for ViTs, concept bottlenecks — but Grad-CAM is the best first stop because it is cheap, intuitive, and works on any model with a conv backbone.

## 2. Grad-CAM intuition

The last conv layer in a CNN outputs feature maps $A^k \in \mathbb{R}^{H' \times W'}$ — one per output channel. Each $A^k$ carries *spatial* information: entry $A^k_{i,j}$ says "channel $k$ fires at pixel $(i,j)$". Channels tend to specialize — fur texture, ear shape, eye region, background foliage, etc.

To explain class $c$, we want to answer: *which feature maps matter for class $c$, and where do they fire?* Grad-CAM's trick: use gradients to score channel importance. A channel that, when turned up slightly, raises $y^c$ a lot has a large gradient — so weight it heavily. Then sum the weighted maps to get a localization heatmap.

That's the whole idea. The math formalizes it.

## 3. Grad-CAM math

Let the model produce logits $y^c$ for class $c$, and let $A^k$ be the activation map of channel $k$ at the chosen conv layer. Grad-CAM defines:

**Step 1 — channel importance weight** (global-average-pool the gradient over space):

$$\alpha^c_k = \frac{1}{H' W'} \sum_{i,j} \frac{\partial y^c}{\partial A^k_{i,j}}$$

**Step 2 — weighted combination + ReLU**:

$$L^c_{\mathrm{Grad\text{-}CAM}} = \mathrm{ReLU}\!\left( \sum_k \alpha^c_k A^k \right)$$

The ReLU keeps only pixels that push the class score *up*. Finally we **bilinearly upsample** $L^c$ to the input resolution and **min-max normalize** to $[0, 1]$ so we can render it as a heatmap.

That's exactly what `utils.gradcam.GradCAM.__call__` implements. Read [utils/gradcam.py](../utils/gradcam.py) — it's under 100 lines of actual logic.

## 4. Load a pretrained EfficientNetV2

Grad-CAM needs a trained model. We use a timm `efficientnetv2_rw_t` with ImageNet-pretrained weights and keep the ImageNet head (1000 classes). This way we don't need to train — and the model still produces meaningful Grad-CAMs because ImageNet includes dogs, cats, birds and many other classes we can photograph.

(If you have the Oxford Pets checkpoint from Notebook 05, you can swap it in by loading `torch.load('checkpoints/05_pets.pt')`.)

In [ ]:
import timm
import torch
import torch.nn.functional as F
import numpy as np

model = timm.create_model('efficientnetv2_rw_t', pretrained=True).to(device).eval()
cfg = model.default_cfg
img_size = cfg.get('input_size', (3, 224, 224))[-1]
mean = cfg.get('mean', (0.485, 0.456, 0.406))
std  = cfg.get('std',  (0.229, 0.224, 0.225))
print('model ready:', sum(p.numel() for p in model.parameters()) / 1e6, 'M params')
print(f'expected input: {img_size}x{img_size}, mean={mean}, std={std}')

## 5. Pick a target layer

For EfficientNetV2, the natural choice is the **last conv before the global pool** (`model.conv_head` in timm). Deeper layers are more class-specific; earlier layers are more texture-y. Pick the last conv unless you have a reason not to.

`utils.gradcam.find_efficientnet_target_layer` tries `conv_head` first, then torchvision's `features[-1]`, then falls back to the last `nn.Conv2d` in the model.

In [ ]:
from utils.gradcam import GradCAM, overlay_cam, find_efficientnet_target_layer

target = find_efficientnet_target_layer(model)
print('target layer:', type(target).__name__, '->', target)

## 6. Grad-CAM on a single image

We'll download a small set of Pet-ish images directly, preprocess them with the timm config, and run one through Grad-CAM.

In [ ]:
from io import BytesIO
from PIL import Image
from torchvision import transforms
import urllib.request, os
from utils.env import data_dir

# Six public-domain / CC pet-ish images (URLs hosted by commons/pytorch samples).
URLS = [
    'https://pytorch.org/tutorials/_static/img/thumbnails/cropped/Transfer-Learning-for-Computer-Vision-Tutorial.png',
    'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/1/14/Labrador_Retriever_portrait.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/b/b6/Felis_catus-cat_on_snow.jpg',
]

cache = os.path.join(data_dir(), 'interp_samples')
os.makedirs(cache, exist_ok=True)

def fetch(url, idx):
    path = os.path.join(cache, f'img_{idx}.jpg')
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            data = r.read()
        with open(path, 'wb') as f:
            f.write(data)
    return Image.open(path).convert('RGB')

images = []
for i, u in enumerate(URLS):
    try:
        images.append(fetch(u, i))
    except Exception as e:
        print(f'[warn] could not fetch {u}: {e}')
print(f'fetched {len(images)} images')

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(img_size + 32),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

tensors = torch.stack([preprocess(im) for im in images]).to(device)
print('batch tensor:', tensors.shape)

with torch.no_grad():
    logits = model(tensors)
pred_ids = logits.argmax(dim=1).cpu().tolist()
print('predicted ImageNet ids:', pred_ids)

Now the actual Grad-CAM call. We use the first image and target the model's own predicted class (`target_class=None` does that automatically).

In [ ]:
cam_gen = GradCAM(model, target)

cam = cam_gen(tensors[0:1])     # pass (1, 3, H, W)
print('cam shape:', cam.shape, 'range:', (cam.min(), cam.max()))

overlay = overlay_cam(tensors[0], cam, alpha=0.5, mean=mean, std=std)
overlay

## 7. Grid of Grad-CAMs (predicted class per image)

Generate Grad-CAM for each of the 6 images and display as a 2x3 grid. A healthy model focuses on the animal's face / body; a concerning signal is a hot-spot on the background.

In [ ]:
from utils.plotting import show_image_grid

overlays = []
titles = []
for i in range(len(images)):
    cam_i = cam_gen(tensors[i:i+1])
    ov = overlay_cam(tensors[i], cam_i, alpha=0.5, mean=mean, std=std)
    overlays.append(np.asarray(ov))           # show_image_grid wants arrays/tensors
    titles.append(f'pred={pred_ids[i]}')

show_image_grid(overlays, titles=titles, ncols=3);

## 8. Grad-CAM++ (library)

**Grad-CAM++** (Chattopadhyay et al., 2017) replaces the simple mean of gradients with a weighted sum involving second-order derivatives. It produces sharper maps and handles images with *multiple instances* of the same class better (e.g. several dogs in one photo). For a thin wrapper we install the `grad-cam` package.

In [ ]:
# !pip install -q grad-cam  # uncomment in Colab if not already installed
try:
    from pytorch_grad_cam import GradCAMPlusPlus, ScoreCAM
    HAVE_GRADCAM_LIB = True
except Exception as e:
    print('pytorch-grad-cam not installed:', e)
    print('Install with: pip install grad-cam')
    HAVE_GRADCAM_LIB = False

if HAVE_GRADCAM_LIB:
    gpp = GradCAMPlusPlus(model=model, target_layers=[target])
    cam_pp = gpp(input_tensor=tensors[0:1])[0]   # (H, W) in [0, 1]
    overlay_pp = overlay_cam(tensors[0], cam_pp, alpha=0.5, mean=mean, std=std)

    # Home-rolled Grad-CAM on the same image
    cam_hr = cam_gen(tensors[0:1])
    overlay_hr = overlay_cam(tensors[0], cam_hr, alpha=0.5, mean=mean, std=std)
    show_image_grid([np.asarray(overlay_hr), np.asarray(overlay_pp)],
                    titles=['Grad-CAM (ours)', 'Grad-CAM++ (lib)'], ncols=2)

## 9. Score-CAM (no gradients)

**Score-CAM** (Wang et al., 2020) skips gradients entirely: it uses each feature map as a *mask* over the input, re-scores the masked image through the model, and weights the maps by the resulting score drop. Pros: no gradient noise, works on non-differentiable models. Cons: expensive (one forward pass per channel — often 1000+ channels).

In [ ]:
if HAVE_GRADCAM_LIB:
    sc = ScoreCAM(model=model, target_layers=[target])
    # Score-CAM is slow — just one image.
    import time
    t0 = time.time()
    cam_sc = sc(input_tensor=tensors[0:1])[0]
    print(f'Score-CAM took {time.time()-t0:.1f}s for one image')
    overlay_sc = overlay_cam(tensors[0], cam_sc, alpha=0.5, mean=mean, std=std)
    overlay_sc

## 10. Debugging workflow — three failure modes

When you look at a misclassified example, Grad-CAM usually reveals one of three problems:

| Symptom | Likely diagnosis | Fix |
| --- | --- | --- |
| Attention on **background** | Data issue — spurious correlation. Model learned that, say, grass $\Rightarrow$ dog. | Fix dataset (balance backgrounds, augment). |
| Attention on **wrong body part** | Architecture / receptive-field issue or wrong target layer. | Try a deeper layer, a larger input size, or a different backbone. |
| Attention is **diffuse / all over** | Under-trained or over-regularized. The model hasn't decided what matters. | Train longer / reduce regularization. |

In the next cell we pick up to 3 images where the model's prediction disagrees with a quick "is it a dog?" sanity check (using ImageNet class 207-268 = dogs, 281-285 = cats). Inspect the attention.

In [ ]:
DOG_IDS = set(range(151, 269))
CAT_IDS = set(range(281, 286)) | {285, 283, 284}

interesting = []
for i, pid in enumerate(pred_ids):
    tag = 'dog' if pid in DOG_IDS else ('cat' if pid in CAT_IDS else 'other')
    interesting.append((i, pid, tag))

for i, pid, tag in interesting:
    print(f'image {i}: predicted id={pid} ({tag})')

# Show Grad-CAM for the two 'other' (suspicious) predictions if any, else first two images.
suspects = [i for i, pid, tag in interesting if tag == 'other'][:2]
if not suspects:
    suspects = [0, 1]

maps = []
cap  = []
for i in suspects:
    cam_i = cam_gen(tensors[i:i+1])
    maps.append(np.asarray(overlay_cam(tensors[i], cam_i, alpha=0.5, mean=mean, std=std)))
    cap.append(f'img {i} pred={pred_ids[i]}')
show_image_grid(maps, titles=cap, ncols=2);

## 11. Complementary lens: raw feature maps

Grad-CAM tells you *where* the model looks for a particular class. Looking at the raw activation maps tells you *what* the individual channels have learned to fire on. We grab the first 16 channels of the target layer for one image and plot them as a 4x4 grid.

In [ ]:
import matplotlib.pyplot as plt

activ = {}
hook = target.register_forward_hook(lambda m, i, o: activ.setdefault('a', o.detach()))
with torch.no_grad():
    _ = model(tensors[0:1])
hook.remove()
fmap = activ['a'][0].cpu()    # (C, H', W')
print('feature map tensor:', fmap.shape)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for k, ax in enumerate(axes.flat):
    ax.imshow(fmap[k], cmap='viridis')
    ax.set_title(f'ch {k}', fontsize=8)
    ax.axis('off')
fig.suptitle('First 16 channels of the target layer', y=0.98)
plt.tight_layout(); plt.show()

## 12. Clean up hooks

`GradCAM` holds two hook handles. When you are done, call `remove_hooks()` so they stop firing on future forward passes.

In [ ]:
cam_gen.remove_hooks()
print('hooks removed -- model is clean for inference')

## 13. Footnote — ViT and attention rollout

Grad-CAM is a *convolution-first* idea. For Vision Transformers, the per-token self-attention matrices provide a natural analogue: **attention rollout** (Abnar & Zuidema, 2020) chains attention across layers to produce a saliency map. `pytorch-grad-cam` actually supports ViTs via a `reshape_transform` callback. We leave that to Notebook 13.

For now: if someone hands you a ViT, reach for attention rollout or integrated gradients, not vanilla Grad-CAM.

## Key Takeaways

- Grad-CAM = channel-importance (from gradients) $\times$ spatial activations, ReLU'd.
- Target the **last conv before the classifier** — `conv_head` on timm EfficientNet.
- A healthy model's CAM lands on the object of interest. Background attention = bad.
- Grad-CAM++ sharpens maps on multi-instance scenes; Score-CAM removes gradient noise.
- Use Grad-CAM to debug: data issues, architecture issues, or under-training — each has a distinct visual signature.
- Always `remove_hooks()` after you're done; the hooks add overhead on every forward pass.

## Exercises

1. Replace `target = model.conv_head` with an earlier block (e.g. `model.blocks[3]` or `model.blocks[5]`). Are the maps coarser or sharper? More or less class-specific?
2. Pick an *incorrect* class to explain. Example: force Grad-CAM to target `target_class=1` (goldfish) on a cat photo. Where does the model look to justify the wrong answer?
3. Replace the backbone with a **ResNet-50** and compare Grad-CAMs on the same image. Do CNN families attend similarly or differently?
4. Add a CIFAR-10 Pet subset from Notebook 09 and generate CAMs for *misclassified* images. Can you identify the failure mode (data / arch / training) from the attention?
5. Implement **Guided Grad-CAM** — multiply the Grad-CAM heatmap by the pixel-space gradient of the class score (aka *guided backprop*). Discuss pros/cons.